# C3 on v4: exp01–exp04 in one session

The same experiments as before, now on release **v4** (the encoder's data), with unchanged code. Two queues, one per T4:
- **GPU 0:** 7B k = 0 (exp03), then 7B k = 1 (exp04)
- **GPU 1:** 0.5B (exp01), 1.5B, 3B (exp03), then 7B k = 3 (exp04)

All zero-shot, definitions on; 7B in 4-bit. Then contextual calibration of the k = 0 runs (exp02, exp03).
Notes: `tasks/C3-prompting-baseline/experiments/`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v4-meetings` dataset attached (**not** the v3 one).

Expected time: about 3 hours (the logs show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder: the v4 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v4"]
assert len(hits) == 1, f"expected exactly one v4 dataset under /kaggle/input, found {hits} : is smi-v4-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Helpers: a queue of runs on one GPU, and waiting while showing progress

In [ ]:
def queue(gpu, jobs):
    """run the jobs one after another on one GPU , in the background ; each run logs to logs/<run_name>.log"""
    commands = []
    for run_name, model, extra in jobs:
        commands.append(
            f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
            f"--run-name {run_name} --model {model} --batch-size 16 {' '.join(extra)} "
            f"> /kaggle/working/logs/{run_name}.log 2>&1"
        )
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    # "; " : a failed run does not stop the next one in the queue
    return subprocess.Popen("; ".join(commands), shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:22} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 5. Run both queues
If the session stops, run this cell again: finished runs are skipped and an interrupted run continues where it stopped.

In [ ]:
SEVEN_B, FOUR_BIT = "Qwen/Qwen2.5-7B-Instruct", ["--load-in-4bit"]
wait([
    queue(0, [
        ("c3_exp03_7b_v4", SEVEN_B, FOUR_BIT),
        ("c3_exp04_7b_k1_v4", SEVEN_B, FOUR_BIT + ["--past-k", "1", "--future-k", "1"]),
    ]),
    queue(1, [
        ("c3_exp01_v4", "Qwen/Qwen2.5-0.5B-Instruct", []),
        ("c3_exp03_1.5b_v4", "Qwen/Qwen2.5-1.5B-Instruct", []),
        ("c3_exp03_3b_v4", "Qwen/Qwen2.5-3B-Instruct", []),
        ("c3_exp04_7b_k3_v4", SEVEN_B, FOUR_BIT + ["--past-k", "3", "--future-k", "3"]),
    ]),
])

## 6. Calibrate the k = 0 runs (exp02, exp03)

In [ ]:
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp01_v4
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_1.5b_v4
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_3b_v4
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_7b_v4

## 7. Results

In [ ]:
print(f"{'run':24} {'macro-F1':>8}  per-class F1")
for name in sorted(os.listdir("/kaggle/working/runs")):
    path = f"/kaggle/working/runs/{name}/metrics.json"
    if not os.path.exists(path):
        print(f"{name:24} (no metrics : see logs/{name}.log)"); continue
    m = json.load(open(path))
    print(f"{name:24} {m['primary']['macro_f1_6']:8.4f}  {m['data_version']}",
          {k: round(v["f1"], 3) for k, v in m["per_class"].items()})

## 8. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_v4_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_v4_runs.zip runs logs
!ls -la c3_v4_runs.zip